In [1]:
# scripts/make_eda.py
"""
Excel → raw CSV → pending-aware clean CSV + long-form EDA

What changes vs before?
- Preserves 'Pending' exactly in new *_status columns (e.g., mc_status, core_status, wg1_status…)
- Adds nullable-boolean analysis columns (True/False/<NA>) mapped from *_status
- any_wg is nullable-boolean: True if any WG True; <NA> if no True but some Pending; else False
- ITC from list stays Yes/No (no Pending), but original Excel ITC text (if any) can be inspected in eda_overview.csv

Usage:
  python scripts/make_eda.py
  python scripts/make_eda.py --input data/raw/participants.xlsx --sheet 0
"""

from __future__ import annotations
from pathlib import Path
from typing import Iterable, Optional
import argparse
import re
import sys

import numpy as np
import pandas as pd


# --------------------------- repo root detection --------------------------- #
def _find_repo_root() -> Path:
    try:
        here = Path(__file__).resolve()
        return here.parent.parent  # …/scripts → repo root
    except NameError:
        cwd = Path.cwd().resolve()
        if (cwd / "data").is_dir() and (cwd / "scripts").is_dir():
            return cwd
        if cwd.name == "scripts" and (cwd.parent / "data").is_dir():
            return cwd.parent
        cur = cwd
        for _ in range(5):
            if (cur / ".git").is_dir() or ((cur / "data").is_dir() and (cur / "scripts").is_dir()):
                return cur
            cur = cur.parent
        return cwd


ROOT = _find_repo_root()
RAW_DIR = ROOT / "data" / "raw"
PROC_DIR = ROOT / "data" / "processed"
OUT_DIR  = ROOT / "outputs"
AUX_DIR  = ROOT / "data"
PROC_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Repo root →", ROOT)


# ------------------------------- helpers ---------------------------------- #
def write_csv(df: pd.DataFrame, path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"Saved → {path.relative_to(ROOT)}")
    return path


def snake_case(name: str) -> str:
    name = str(name).replace("\u00A0", " ")
    name = re.sub(r"[^\w\s\-]+", " ", name)
    name = " ".join(name.split()).strip().lower().replace("-", " ")
    return re.sub(r"\s+", "_", name)


def clean_text_series(s: pd.Series) -> pd.Series:
    out = (
        s.astype(str)
         .str.replace(r"\*", "", regex=True)
         .str.replace("\u00A0", " ", regex=False)
         .str.replace(r"\s+", " ", regex=True)
         .str.strip()
    )
    return out.replace({"": np.nan, "nan": np.nan})


_YN_PATTERNS = [re.compile(p, re.I) for p in (
    r"^wg\s*\d+",
    r"^wg\d+",
    r"^is_",
    r"\bmc\b|\bmc_member\b",
    r"\bcore\b|\bcore_group\b",
    r"^wg_member$",
)]


def looks_like_yn(names: Iterable[str]) -> list[bool]:
    return [any(p.search(str(c).lower()) for p in _YN_PATTERNS) for c in names]


def status_from_tokens(s: pd.Series) -> pd.Series:
    """
    Map common tokens to 'Yes' / 'No' / 'Pending' or NA.
    Keeps unknown tokens as-is (upper/lower normalized) so nothing is lost.
    """
    raw = s.astype(str).str.strip()
    low = raw.str.lower()

    out = pd.Series(index=s.index, dtype="object")

    # Yes-ish
    yes_mask = (
        low.isin({"y", "yes", "true", "1", "member", "x"})
        | low.str.match(r"^\s*yes\b", na=False)
    )
    # No-ish
    no_mask = low.isin({"n", "no", "false", "0"})

    # Pending-ish
    pend_mask = (
        low.str.contains(r"\bpending\b|\btbc\b|\bawaiting\b", na=False)
    )

    out[yes_mask & ~pend_mask] = "Yes"
    out[no_mask] = "No"
    out[pend_mask] = "Pending"

    # keep other tokens but title-case them
    others = out.isna()
    out[others] = raw[others].where(raw[others].ne(""), np.nan)
    out = out.map(lambda v: v.title() if isinstance(v, str) else v)

    return out


def status_to_nullable_bool(s: pd.Series) -> pd.Series:
    """
    'Yes' -> True ; 'No' -> False ; 'Pending'/NA/others -> <NA> (pandas BooleanDtype)
    """
    return s.map({"Yes": True, "No": False}).astype("boolean")


def gentle_type_infer(s: pd.Series) -> pd.Series:
    """Two-pass, conservative typing: numeric then datetime."""
    if s.dtype == "object":
        raw = s.astype(str).str.replace(",", "").str.strip()
        looks_num = raw.str.match(r"^-?\d+(\.\d+)?$", na=False)
        if looks_num.mean() >= 0.6:
            return pd.to_numeric(raw, errors="coerce")

    if s.dtype == "object" or s.dtype.kind in "Mm":
        if s.dtype.kind in "Mm":
            return s
        sample = s.astype(str).str.lower()
        looks_date = sample.str.contains(
            r"\d{1,4}[-/]\d{1,2}[-/]\d{1,4}|jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec",
            regex=True, na=False
        )
        if looks_date.mean() >= 0.5:
            parsed = pd.to_datetime(s, errors="coerce", dayfirst=True)
            if parsed.notna().mean() >= 0.5:
                return parsed
    return s


def clean_country_column(df: pd.DataFrame) -> pd.DataFrame:
    """Add 'country_clean' if a 'country' column exists."""
    matches = [c for c in df.columns if c.lower() == "country"]
    if not matches:
        return df
    col = matches[0]
    df["country_clean"] = (
        df[col].astype(str)
              .str.replace(r"\(.*?\)", "", regex=True)
    )
    df["country_clean"] = clean_text_series(df["country_clean"])

    aliases = {
        "uk": "United Kingdom", "united kingdom": "United Kingdom", "great britain": "United Kingdom",
        "czech republic": "Czechia",
        "turkey": "Türkiye", "turkiye": "Türkiye",
        "north macedonia": "North Macedonia", "macedonia": "North Macedonia",
        "bosnia and herzegovina": "Bosnia and Herzegovina",
        "moldova": "Moldova", "russia": "Russia",
        "ivory coast": "Côte d'Ivoire", "cote d ivoire": "Côte d'Ivoire",
        "republic of kosovo": "Kosovo", "kosovo": "Kosovo",
        "republic of serbia": "Serbia",
        "republic of north macedonia": "North Macedonia",
    }
    low = df["country_clean"].str.lower()
    mask = low.isin(aliases)
    df.loc[mask, "country_clean"] = low.map(aliases)
    return df


def value_counts_preview(s: pd.Series, n: int = 10) -> str:
    vc = s.astype("string").fillna("<NA>").value_counts(dropna=False).head(n)
    return "; ".join(f"{k}: {int(v)}" for k, v in vc.items())


def mark_section(df: pd.DataFrame, name: str) -> pd.DataFrame:
    out = df.copy()
    out.insert(0, "section", name)
    return out


# ---------------- ITC support: load list + normalise for matching ------------ #
def _norm_country(s: str) -> str:
    s = (str(s) or "").strip()
    s = re.sub(r"\s*\([^)]*\)\s*$", "", s)
    s = s.replace("’", "'")
    s = " ".join(s.split()).lower()
    s = s.replace("republic of ", "")
    aliases_lc = {
        "czech republic": "czechia",
        "macedonia": "north macedonia",
        "turkey": "türkiye",
        "turkiye": "türkiye",
        "türkiye": "türkiye",
        "uk": "united kingdom",
        "cote d'ivoire": "côte d'ivoire",
    }
    return aliases_lc.get(s, s)


def load_itc_set(path: Path) -> set[str]:
    raw = path.read_text(encoding="utf-8").splitlines()
    items = [_norm_country(x) for x in raw if str(x).strip()]
    return set(items)


# -------------------------- CLI / input selection -------------------------- #
def find_default_excel(raw_dir: Path) -> Optional[Path]:
    preferred = raw_dir / "CA23107participant list.xlsx"
    if preferred.exists():
        return preferred
    files = sorted(list(raw_dir.glob("*.xlsx")) + list(raw_dir.glob("*.xls")))
    return files[0] if files else None


def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(description="Build data_raw.csv, data_clean.csv, eda_overview.csv (pending-aware)")
    p.add_argument("--input", "-i", type=str, default=None, help="Excel file (default: CA23107… or first .xlsx in data/raw/)")
    p.add_argument("--sheet", "-s", default=0, help="Sheet index (0) or name (str). Default: 0")
    p.add_argument("--topn", type=int, default=25, help="Top-N values in EDA previews (default 25)")
    if "ipykernel" in sys.modules or "IPython" in sys.modules:
        args, _ = p.parse_known_args()
    else:
        args = p.parse_args()
    return args


# -------------------------------- pipeline --------------------------------- #
def main() -> None:
    args = parse_args()

    # Resolve input Excel path
    if args.input:
        excel_path = Path(args.input)
        if not excel_path.is_absolute():
            excel_path = (ROOT / excel_path).resolve()
    else:
        excel_path = find_default_excel(RAW_DIR) or Path()

    if not excel_path.exists():
        print("ERROR: Excel not found.")
        print(f"- Looked for: {RAW_DIR / 'CA23107participant list.xlsx'}")
        print(f"- Or first .xlsx/.xls in: {RAW_DIR}")
        print("Or pass: python scripts/make_eda.py --input data/raw/yourfile.xlsx")
        sys.exit(1)

    sheet = args.sheet
    top_n = args.topn

    # Friendly path print
    rel = excel_path
    try:
        rel = excel_path.relative_to(ROOT)
    except Exception:
        pass
    print(f"Reading Excel: {rel}")

    # 1) Read Excel as-is
    df_raw = pd.read_excel(excel_path, sheet_name=sheet)

    # 2) Save verbatim copy
    write_csv(df_raw, RAW_DIR / "data_raw.csv")

    # 3) Detect Y/N-like columns BEFORE cleaning
    yn_flags_before = looks_like_yn(df_raw.columns)
    yn_candidates_raw = [c for c, is_yn in zip(df_raw.columns, yn_flags_before) if is_yn]
    yn_before_df = pd.DataFrame({
        "original_name": yn_candidates_raw,
        "normalized_name_if_any": [snake_case(c) for c in yn_candidates_raw],
        "dtype_raw": [str(df_raw[c].dtype) for c in yn_candidates_raw],
        "top_values_preview": [value_counts_preview(df_raw[c], n=10) for c in yn_candidates_raw],
    })

    # 4) Light cleaning
    df = df_raw.copy()

    # 4a) normalize headers
    old_to_new = {c: snake_case(c) for c in df.columns}
    df.rename(columns=old_to_new, inplace=True)

    # 4b) clean text columns
    obj_cols = df.select_dtypes(include="object").columns
    if len(obj_cols):
        df[obj_cols] = df[obj_cols].apply(clean_text_series)

    # 4c) country helper (adds 'country_clean' if 'country' exists)
    df = clean_country_column(df)

    # 4d) ITC countries (Yes/No) using data/itc_countries.txt if present
    itc_path = AUX_DIR / "itc_countries.txt"
    base_col = "country_clean" if "country_clean" in df.columns else ("country" if "country" in df.columns else None)
    if itc_path.exists() and base_col:
        itc_set = load_itc_set(itc_path)
        df["itc_countries"] = df[base_col].map(lambda x: "Yes" if _norm_country(x) in itc_set else "No")
        print(f"Loaded ITC list ({len(itc_set)} names) → added 'itc_countries' (Yes/No).")
    elif base_col:
        df["itc_countries"] = "No"  # deterministic column even if file missing

    # 4e) PENDING-AWARE flags
    # Find likely Y/N flag columns AFTER header cleaning
    yn_flags_after = looks_like_yn(df.columns)
    yn_cols_after = [c for c, is_yn in zip(df.columns, yn_flags_after) if is_yn]

    # 4e-i) Create *_status columns with 'Yes'/'No'/'Pending'/NA
    status_cols = []
    for c in yn_cols_after:
        status_col = f"{c}_status"
        df[status_col] = status_from_tokens(df[c])
        status_cols.append(status_col)

    # 4e-ii) Replace original flag columns with nullable booleans mapped from *_status
    for c in yn_cols_after:
        status_col = f"{c}_status"
        df[c] = status_to_nullable_bool(df[status_col])

    # 4e-iii) Derive any_wg as nullable boolean from all WG columns
    wg_cols = [c for c in yn_cols_after if re.match(r"(?i)^wg\d+", c)]
    if wg_cols:
        any_true = df[wg_cols].apply(lambda r: bool(r.fillna(False).any()), axis=1)
        any_pending = (
            df[[f"{c}_status" for c in wg_cols if f"{c}_status" in df]]
            .eq("Pending").any(axis=1)
        )
        df["any_wg"] = pd.Series(np.where(any_true, True,
                                          np.where(any_pending, pd.NA, False)),
                                 dtype="boolean")
    elif "any_wg" not in df.columns:
        df["any_wg"] = pd.Series([False]*len(df), dtype="boolean")

    # 5) Conservative type inference for non-status columns
    # (Keep *_status as string; leave boolean dtypes intact)
    skip_cols = set(status_cols + yn_cols_after + ["any_wg", "itc_countries"])
    for c in df.columns:
        if c in skip_cols:
            continue
        df[c] = gentle_type_infer(df[c])

    # 6) Save cleaned CSV
    write_csv(df, PROC_DIR / "data_clean.csv")

    # 7) Build long-form EDA CSV
    eda_parts: list[pd.DataFrame] = []

    # 7a) dtype changes (before vs after)
    dtype_changes = pd.DataFrame({
        "original_name": list(df_raw.columns),
        "cleaned_name": [old_to_new.get(c, c) for c in df_raw.columns],
        "dtype_raw": [str(df_raw[c].dtype) for c in df_raw.columns],
        "dtype_final": [str(df[old_to_new.get(c, c)].dtype) if old_to_new.get(c, c) in df.columns else "<missing>" for c in df_raw.columns],
    })
    dtype_changes["changed"] = dtype_changes["dtype_raw"] != dtype_changes["dtype_final"]
    eda_parts.append(mark_section(dtype_changes, "dtype_changes"))

    # 7b) Y/N detection before cleaning
    if not yn_before_df.empty:
        eda_parts.append(mark_section(yn_before_df, "yn_detection_before"))

    # 7c) Y/N summary after cleaning (pending-aware)
    if yn_cols_after:
        rows = []
        for c in yn_cols_after:
            s = df[c]  # nullable boolean
            sc = f"{c}_status"
            rows.append({
                "flag_col": c,
                "status_col": sc,
                "dtype_flag": str(s.dtype),
                "true_count": int(s.fillna(False).sum()),
                "false_count": int((~s.fillna(False)).sum() - int(s.isna().sum())),  # strictly False, excluding NAs
                "na_count": int(s.isna().sum()),
                "status_preview": value_counts_preview(df[sc], n=10)
            })
        eda_parts.append(mark_section(pd.DataFrame(rows), "yn_after_pending_aware"))

    # 7d) columns summary
    cols_summary = pd.DataFrame({
        "column": df.columns,
        "dtype": [str(df[c].dtype) for c in df.columns],
        "non_null": [int(df[c].notna().sum()) for c in df.columns],
        "missing": [int(df[c].isna().sum()) for c in df.columns],
        "unique": [int(df[c].nunique(dropna=True)) for c in df.columns],
        "example": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
    }).sort_values("column")
    eda_parts.append(mark_section(cols_summary, "columns_summary"))

    # 7e) numeric summary (long format)
    num_cols = df.select_dtypes(include=[np.number, "boolean"]).columns
    if len(num_cols):
        desc_long = (
            df[num_cols].describe(include="all")
                        .T.reset_index().rename(columns={"index": "column"})
                        .melt(id_vars="column", var_name="metric", value_name="value")
        )
        eda_parts.append(mark_section(desc_long, "numeric_summary"))

    # 7f) top values for object/category columns (incl *_status)
    cat_cols = df.select_dtypes(include=["object", "category"]).columns
    if len(cat_cols):
        rows = []
        for c in cat_cols:
            vc = df[c].astype("string").fillna("<NA>").value_counts(dropna=False).head(top_n)
            rows.extend({"column": c, "value": k, "count": int(v)} for k, v in vc.items())
        eda_parts.append(mark_section(pd.DataFrame(rows), "top_values"))

    # 7g) country counts (if present)
    country_col = [c for c in df.columns if c == "country_clean"] or [c for c in df.columns if c == "country"]
    if country_col:
        country_counts = (
            df[country_col[0]].astype("string").fillna("<NA>")
              .value_counts(dropna=False).rename_axis("country")
              .reset_index(name="count")
        )
        eda_parts.append(mark_section(country_counts, "countries_counts"))

    eda_overview = pd.concat(eda_parts, ignore_index=True, sort=False) if eda_parts else pd.DataFrame({"section":[]})
    write_csv(eda_overview, OUT_DIR / "eda_overview.csv")

    print("\nAll files written under:")
    print(" -", RAW_DIR.relative_to(ROOT))   # data_raw.csv
    print(" -", PROC_DIR.relative_to(ROOT))  # data_clean.csv (pending-aware)
    print(" -", OUT_DIR.relative_to(ROOT))   # eda_overview.csv


if __name__ == "__main__":
    main()


Repo root → C:\Users\James\Documents\GitHub\evidence-map-agrifood
Reading Excel: data\raw\CA23107participant list.xlsx
Saved → data\raw\data_raw.csv
Loaded ITC list (25 names) → added 'itc_countries' (Yes/No).
Saved → data\processed\data_clean.csv
Saved → outputs\eda_overview.csv

All files written under:
 - data\raw
 - data\processed
 - outputs
